In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("yaml") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyyaml"])

In [ ]:
from pathlib import Path
import subprocess
import sys
from IPython.display import Image as DisplayImage, display
def locate_repo_root():
    starts = [Path.cwd().resolve(), Path("/kaggle/working/beyond-iid")]
    seen = set()
    for start in starts:
        for candidate in (start, *start.parents):
            for root in (candidate, candidate / "beyond-iid"):
                root = root.resolve()
                if root in seen:
                    continue
                seen.add(root)
                if (root / "shared/pacs.py").is_file() and (root / "domain-generalization/train.py").is_file():
                    return root
    raise FileNotFoundError(
        "Could not locate beyond-iid. Clone it under /kaggle/working or run this notebook from the repository."
    )
REPO_ROOT = locate_repo_root()
TASK_DIR = REPO_ROOT / "domain-generalization"
TASK2_DIR = REPO_ROOT / "domain-adaptation"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from shared.pacs import find_pacs_root
resolved_pacs_root = find_pacs_root(PACS_ROOT)
task2_checkpoint = REPO_ROOT / "shared/checkpoints/source_only_pacs_sketch_seed6304.pt"
task2_history = TASK2_DIR / "results/histories/source_only.csv"
if not task2_checkpoint.is_file() or not task2_history.is_file():
    print("Task 2 ERM artifact is missing; regenerating only Source-only in an isolated process.", flush=True)
    recovery_code = r'''
import sys
from pathlib import Path
repo_root = Path(sys.argv[1]).resolve()
task_dir = repo_root / "domain-adaptation"
pacs_root = Path(sys.argv[2]).resolve()
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(task_dir))
import train as task2_train
assert Path(task2_train.__file__).resolve() == task_dir / "train.py", task2_train.__file__
task2_train.MAIN_METHODS[:] = ["source_only"]
task2_train.train_suite(pacs_root, task_dir, include_study=False, force=False)
'''
    subprocess.run(
        [sys.executable, "-c", recovery_code, str(REPO_ROOT), str(resolved_pacs_root)],
        check=True,
    )
assert task2_checkpoint.is_file(), f"Task 2 checkpoint was not created: {task2_checkpoint}"
assert task2_history.is_file(), f"Task 2 history was not created: {task2_history}"
for module_name in (
    "train", "configs", "models", "methods", "training", "evaluation",
    "selection", "evaluate_source", "evaluate_sketch",
):
    module = sys.modules.get(module_name)
    module_file = getattr(module, "__file__", None) if module else None
    if module_file and TASK_DIR not in Path(module_file).resolve().parents:
        raise RuntimeError(
            f"{module_name!r} was already imported from {module_file}. "
            "Restart the kernel and run this notebook from the first cell."
        )
for path in (str(REPO_ROOT), str(TASK_DIR)):
    if path in sys.path:
        sys.path.remove(path)
    sys.path.insert(0, path)
import train as task3_train
assert Path(task3_train.__file__).resolve() == TASK_DIR / "train.py", task3_train.__file__
from evaluate_source import evaluate_source_suite
from evaluate_sketch import evaluate_sketch_suite
train_suite = task3_train.train_suite
print({
    "repo_root": str(REPO_ROOT),
    "task_dir": str(TASK_DIR),
    "pacs_root": str(resolved_pacs_root),
    "train_module": str(Path(task3_train.__file__).resolve()),
    "task2_checkpoint_ready": task2_checkpoint.is_file(),
})

In [ ]:
training = train_suite(resolved_pacs_root, TASK_DIR, include_study=True, force=False)
display(training["run_manifest"])
display(training["histories"].groupby("method").tail(1).round(4))
display(DisplayImage(filename=str(TASK_DIR / "results/main_training_curves.png"), width=1050))
display(DisplayImage(filename=str(TASK_DIR / "results/sam_radius_training_curves.png"), width=900))

In [ ]:
source_results = evaluate_source_suite(resolved_pacs_root, TASK_DIR)
display(source_results["comparison"].round(4))
display(source_results["separability"].round(4))
display(source_results["sharpness"].round(5))
display(DisplayImage(filename=str(TASK_DIR / "results/source_diagnostics.png"), width=900))

In [ ]:
CONFIGURATIONS_LOCKED = False
assert CONFIGURATIONS_LOCKED, (
    "Set CONFIGURATIONS_LOCKED=True only after every Task 3 decision is frozen. "
    "The next cell is the first Task 3 operation that loads Sketch."
)

In [ ]:
final = evaluate_sketch_suite(
    resolved_pacs_root, TASK_DIR, configurations_locked=CONFIGURATIONS_LOCKED
)
display(final["main_comparison"].round(4))
display(final["study"].round(4))
display(final["cross_task"].round(4))
display(DisplayImage(filename=str(TASK_DIR / "results/sketch_per_class_accuracy_changes.png"), width=950))
display(DisplayImage(filename=str(TASK_DIR / "results/sketch_confusion_matrices.png"), width=1200))
display(DisplayImage(filename=str(TASK_DIR / "results/selected_sketch_failure_cases.png"), width=1000))